# Iris Flower Classification using Machine Learning

**Objective:** Predict whether an iris flower is Setosa, Versicolor, or Virginica from sepal and petal measurements.

**Tech stack:** Python, pandas, NumPy, Matplotlib, Seaborn, scikit-learn.

This notebook covers dataset loading, EDA, visualisation, feature discussion, train/test splitting, four classifiers, evaluation, and best-model selection.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_fscore_support, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the Iris dataset

In [ ]:
# Load the dataset directly from scikit-learn (no download required)
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

# Create a combined DataFrame for EDA and visualisation
df = X.copy()
df["species"] = y.map(dict(enumerate(iris.target_names)))

print("Dataset loaded successfully.")
display(df.head())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset dimensions
print("Dataset shape:", df.shape)

# Data types
print("\\nData types:")
print(df.dtypes)

# Missing-value check
print("\\nMissing values:")
print(df.isnull().sum())

# Descriptive statistics
print("\\nDescriptive statistics:")
display(df.describe())

# Class distribution
print("\\nSpecies counts:")
display(df["species"].value_counts())

## 4. Data visualisation

### 4.1 Pairplot – feature relationships by species

In [ ]:
# Pairplot compares every pair of features and colours points by species
sns.pairplot(df, hue="species", diag_kind="hist", corner=True)
plt.suptitle("Iris Feature Relationships by Species", y=1.02)
plt.show()

### 4.2 Boxplots – distribution of each feature

In [ ]:
feature_names = iris.feature_names

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, feature in zip(axes, feature_names):
    sns.boxplot(data=df, x="species", y=feature, ax=ax)
    ax.set_title(f"{feature} by species")
    ax.set_xlabel("Species")
    ax.set_ylabel(feature)

plt.tight_layout()
plt.show()

### 4.3 Correlation heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(X.corr(), annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Between Iris Features")
plt.tight_layout()
plt.show()

## 5. Feature selection discussion

The pairplot and boxplots help compare how well each measurement separates the three species.

- **Petal length and petal width** are generally the most discriminative features. Setosa is especially distinct, while Versicolor and Virginica show some overlap.
- **Sepal length** can provide additional information, but the classes overlap more.
- **Sepal width** is often less discriminative when used alone.

For this baseline project, all four features are retained so that the models can learn from the complete dataset. Feature importance and visual separation can be discussed in the final report.

## 6. Train/test split

In [ ]:
# 80% training data and 20% testing data.
# stratify=y keeps the class proportions similar in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("\\nTraining class distribution:")
print(y_train.value_counts().sort_index())

## 7. Train classification models

Four algorithms are compared:

1. Logistic Regression – uses a linear decision boundary.
2. K-Nearest Neighbours (KNN) – predicts using nearby training samples.
3. Decision Tree – learns rule-based splits.
4. Random Forest – combines multiple decision trees.

Scaling is applied to Logistic Regression and KNN using a Pipeline. This ensures the scaler is fitted only on training data.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "K-Nearest Neighbours": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
}

# Train each model and store its predictions
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    print(f"{name}: training complete")

## 8. Model evaluation

In [ ]:
# Accuracy, weighted precision, weighted recall, and weighted F1-score
results = []

for name, y_pred in predictions.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (weighted)": precision,
        "Recall (weighted)": recall,
        "F1-score (weighted)": f1
    })

results_df = pd.DataFrame(results).sort_values(
    by=["Accuracy", "F1-score (weighted)"], ascending=False
).reset_index(drop=True)

display(results_df.round(4))

### 8.1 Confusion matrices

In [ ]:
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=iris.target_names
    )
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix – {name}")
    plt.tight_layout()
    plt.show()

### 8.2 Classification reports

In [ ]:
for name, y_pred in predictions.items():
    print(f"\\nClassification Report – {name}")
    print(classification_report(
        y_test, y_pred,
        target_names=iris.target_names,
        zero_division=0
    ))

## 9. Compare model performance

In [ ]:
plot_df = results_df.set_index("Model")[
    ["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"]
]

plot_df.plot(kind="bar", figsize=(12, 6), ylim=(0, 1.05))
plt.title("Performance Comparison of Classification Models")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=25, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 10. Best-performing model

In [ ]:
# Select the best model using accuracy first, then weighted F1-score.
best_row = results_df.iloc[0]
best_model_name = best_row["Model"]
best_model = models[best_model_name]

print("Best-performing model:", best_model_name)
print(f"Accuracy: {best_row['Accuracy']:.4f}")
print(f"Weighted precision: {best_row['Precision (weighted)']:.4f}")
print(f"Weighted recall: {best_row['Recall (weighted)']:.4f}")
print(f"Weighted F1-score: {best_row['F1-score (weighted)']:.4f}")

print("\\nJustification:")
print(
    "This model ranks highest on the held-out test set using accuracy, "
    "with weighted F1-score used to break ties. Review its confusion matrix "
    "and per-class classification report before making a final conclusion."
)

## 11. Feature importance (tree-based model)

Random Forest provides a useful indication of how much each feature contributes to its predictions. These importances are model-specific and should not be interpreted as proof of causation.

In [ ]:
rf_model = models["Random Forest"]
importance = pd.Series(
    rf_model.feature_importances_,
    index=iris.feature_names
).sort_values(ascending=False)

display(importance.rename("Importance").to_frame())

importance.sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 12. Test with a new flower measurement

Enter measurements in centimetres in the same order as the dataset features: sepal length, sepal width, petal length, petal width.

In [ ]:
# Example measurements; replace these values with a new flower's measurements.
new_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=iris.feature_names
)

predicted_class = best_model.predict(new_flower)[0]
print("Predicted iris species:", iris.target_names[predicted_class])

## 13. Conclusion

This project used the built-in Iris dataset to build and compare four classification models. EDA and visualisations were used to understand the data and examine feature separation. Models were evaluated on a stratified 20% test set using accuracy, confusion matrices, precision, recall, and F1-score.

The best model is selected programmatically from the test-set comparison. Report the model name and its measured scores from the executed notebook rather than assuming one algorithm is always best.

**Important:** The Iris dataset is small and clean, so high test accuracy is possible. For a stronger evaluation, consider cross-validation and repeat the experiment with different random splits.

## 14. References

- Scikit-learn Iris dataset: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html
- Scikit-learn model selection: https://scikit-learn.org/stable/model_selection.html
- Scikit-learn metrics: https://scikit-learn.org/stable/modules/model_evaluation.html
- Seaborn pairplot: https://seaborn.pydata.org/generated/seaborn.pairplot.html
- Seaborn boxplot: https://seaborn.pydata.org/generated/seaborn.boxplot.html